# 13_translation: What Is Preserved?

**Lab report:** Invariant Specification Demystifies Translation

Notebook 00 asked whether GLOSSOPETRAE exposes a meaningful object called a *specification*.

Notebook 07 decomposed that object into candidate components:

- grammar
- lexicon
- phonology
- script
- semantics
- translation mappings
- evolution rules

This notebook investigates the central lab-report claim:

> Translation changes representation, not specification.

The working question is:

> What survives translation?

## 1. Setup

Clone or reuse the upstream GLOSSOPETRAE repository.

In [ ]:
from pathlib import Path
import os
import re
import json
import csv
import subprocess
from collections import Counter, defaultdict

ROOT = Path.cwd()
UPSTREAM_URL = "https://github.com/elder-plinius/GLOSSOPETRAE.git"
UPSTREAM_DIR = ROOT / "GLOSSOPETRAE"

if not UPSTREAM_DIR.exists():
    subprocess.run(["git", "clone", UPSTREAM_URL, str(UPSTREAM_DIR)], check=True)
else:
    print("Upstream repo already present:", UPSTREAM_DIR)

FIGURES = ROOT / "figures"
DATA = ROOT / "data"
FIGURES.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)

print("Repository exists:", UPSTREAM_DIR.exists())

## 2. Search for translation artifacts

Locate files and source references that indicate how translation is represented.

Important distinction:

```text
Language A → Language B
```

is different from:

```text
Language A → Meaning / concept → Language B
```

The second form is closer to an invariant-specification model.

In [ ]:
translation_terms = [
    "translation",
    "translate",
    "mapping",
    "correspondence",
    "meaning",
    "semantic",
    "concept",
    "gloss",
    "phrase",
    "sentence",
    "parallel",
    "alignment",
    "cognate",
]

TEXT_SUFFIXES = {".py", ".md", ".txt", ".yaml", ".yml", ".json", ".toml", ".js", ".ts", ".html", ".css", ".csv", ".tsv"}

path_hits = []
content_hits = []

for path in UPSTREAM_DIR.rglob("*"):
    if ".git" in path.parts or path.is_dir():
        continue

    rel = path.relative_to(UPSTREAM_DIR)
    lower_path = str(rel).lower()

    path_terms = [t for t in translation_terms if t in lower_path]
    if path_terms:
        path_hits.append({
            "path": str(rel),
            "suffix": path.suffix,
            "hits": ", ".join(path_terms),
            "size_bytes": path.stat().st_size if path.exists() else None,
        })

    if path.suffix.lower() not in TEXT_SUFFIXES:
        continue

    try:
        text = path.read_text(errors="ignore")
    except Exception:
        continue

    lower = text.lower()
    for term in translation_terms:
        count = lower.count(term)
        if count:
            content_hits.append({
                "path": str(rel),
                "term": term,
                "count": count,
            })

len(path_hits), len(content_hits)

In [ ]:
try:
    import pandas as pd

    path_df = pd.DataFrame(path_hits).sort_values(["hits", "path"])
    content_df = pd.DataFrame(content_hits)

    display(path_df.head(80))

    term_summary = (
        content_df
        .groupby("term", as_index=False)["count"]
        .sum()
        .sort_values("count", ascending=False)
    )
    display(term_summary)

    path_df.to_csv(DATA / "13_translation_path_hits.csv", index=False)
    term_summary.to_csv(DATA / "13_translation_term_summary.csv", index=False)
except Exception:
    print("Path hits:")
    for row in path_hits[:80]:
        print(row)
    print("\\nTerm counts:")
    print(Counter([r["term"] for r in content_hits]))

## 3. Extract translation snippets

Look around high-value words to see whether the repo treats translation as word replacement, meaning preservation, mapping, correspondence, or generation from a shared specification.

In [ ]:
def snippets_for(term, max_snippets=20, radius=180):
    rows = []
    for path in UPSTREAM_DIR.rglob("*"):
        if ".git" in path.parts or path.is_dir() or path.suffix.lower() not in TEXT_SUFFIXES:
            continue

        try:
            text = path.read_text(errors="ignore")
        except Exception:
            continue

        lower = text.lower()
        start_search = 0
        while len(rows) < max_snippets:
            idx = lower.find(term.lower(), start_search)
            if idx < 0:
                break

            start = max(0, idx - radius)
            end = min(len(text), idx + len(term) + radius)
            snippet = text[start:end].replace("\\n", " ")
            rows.append({
                "term": term,
                "path": str(path.relative_to(UPSTREAM_DIR)),
                "snippet": snippet,
            })
            start_search = idx + len(term)

        if len(rows) >= max_snippets:
            break

    return rows

snippet_terms = ["translation", "translate", "meaning", "semantic", "concept", "mapping", "correspondence"]
snippets = []
for term in snippet_terms:
    snippets.extend(snippets_for(term, max_snippets=12))

try:
    import pandas as pd
    snippets_df = pd.DataFrame(snippets)
    display(snippets_df)
    snippets_df.to_csv(DATA / "13_translation_snippets.csv", index=False)
except Exception:
    for s in snippets:
        print("\\n---", s["term"], s["path"])
        print(s["snippet"])

## 4. Candidate translation files

Find small files that may contain examples, mappings, glosses, prompts, or outputs.

In [ ]:
candidate_translation_files = []

for path in UPSTREAM_DIR.rglob("*"):
    if ".git" in path.parts or path.is_dir() or path.suffix.lower() not in TEXT_SUFFIXES:
        continue

    rel = path.relative_to(UPSTREAM_DIR)
    lower = str(rel).lower()

    path_score = sum(1 for term in translation_terms if term in lower)

    try:
        text = path.read_text(errors="ignore")
    except Exception:
        text = ""

    content_score = sum(text.lower().count(term) for term in translation_terms)

    if path_score or content_score:
        candidate_translation_files.append({
            "path": str(rel),
            "suffix": path.suffix,
            "size_bytes": path.stat().st_size,
            "path_score": path_score,
            "content_score": content_score,
            "total_score": path_score * 10 + content_score,
        })

candidate_translation_files = sorted(
    candidate_translation_files,
    key=lambda r: (-r["total_score"], r["size_bytes"], r["path"])
)

try:
    import pandas as pd
    trans_files_df = pd.DataFrame(candidate_translation_files)
    display(trans_files_df.head(60))
    trans_files_df.to_csv(DATA / "13_candidate_translation_files.csv", index=False)
except Exception:
    for row in candidate_translation_files[:60]:
        print(row)

## 5. Observable-change table

This table is the notebook's first experimental frame.

For a translation event, mark whether each property changes, stays invariant, or remains unresolved.

Edit this after running or inspecting actual translation examples.

In [ ]:
observable_change_rows = [
    {
        "property": "surface words",
        "expected_change": "changes",
        "observed": "unresolved",
        "notes": "Translation normally changes lexical forms."
    },
    {
        "property": "pronunciation",
        "expected_change": "changes",
        "observed": "unresolved",
        "notes": "Speech representation should change when phonology changes."
    },
    {
        "property": "script / glyphs",
        "expected_change": "changes",
        "observed": "unresolved",
        "notes": "Writing system may change while preserving mapped content."
    },
    {
        "property": "grammar",
        "expected_change": "partial",
        "observed": "unresolved",
        "notes": "Grammar may change while preserving the intended proposition."
    },
    {
        "property": "meaning / concept",
        "expected_change": "preserved",
        "observed": "unresolved",
        "notes": "Meaning is the strongest invariant candidate."
    },
    {
        "property": "correspondence",
        "expected_change": "preserved",
        "observed": "unresolved",
        "notes": "Mappings may persist even when forms change."
    },
    {
        "property": "lineage",
        "expected_change": "not applicable or preserved",
        "observed": "unresolved",
        "notes": "Lineage matters more for descendants than translation."
    },
]

try:
    import pandas as pd
    observable_df = pd.DataFrame(observable_change_rows)
    display(observable_df)
    observable_df.to_csv(DATA / "13_observable_change_template.csv", index=False)
except Exception:
    for row in observable_change_rows:
        print(row)

## 6. Correspondence extraction scaffold

If the repo exposes translation pairs, mappings, or glosses, load them into a common table.

Target schema:

| source_form | target_form | meaning | source_language | target_language | evidence_path |
|---|---|---|---|---|---|

This scaffold includes a lightweight parser for obvious JSON/YAML/CSV structures but leaves space for manual inspection.

In [ ]:
def try_load_structured(path):
    suffix = path.suffix.lower()
    try:
        if suffix == ".json":
            return json.loads(path.read_text(errors="ignore"))
        if suffix in {".yaml", ".yml"}:
            try:
                import yaml
                return yaml.safe_load(path.read_text(errors="ignore"))
            except Exception:
                return None
        if suffix == ".csv":
            with path.open(newline="", errors="ignore") as f:
                return list(csv.DictReader(f))
        if suffix == ".tsv":
            with path.open(newline="", errors="ignore") as f:
                return list(csv.DictReader(f, delimiter="\\t"))
    except Exception:
        return None
    return None

def flatten_records(obj, path=""):
    records = []
    if isinstance(obj, dict):
        # Heuristic: dictionary with translation-like keys.
        keys = {str(k).lower() for k in obj.keys()}
        if keys & {"source", "target", "translation", "meaning", "gloss", "concept"}:
            records.append(obj)
        for k, v in obj.items():
            records.extend(flatten_records(v, path=f"{path}.{k}" if path else str(k)))
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            records.extend(flatten_records(item, path=f"{path}[{i}]"))
    return records

structured_records = []

for row in candidate_translation_files[:80]:
    path = UPSTREAM_DIR / row["path"]
    obj = try_load_structured(path)
    if obj is None:
        continue
    records = flatten_records(obj)
    for rec in records:
        if isinstance(rec, dict):
            structured_records.append({
                "evidence_path": row["path"],
                "raw_record": json.dumps(rec, ensure_ascii=False)[:1000],
            })

try:
    import pandas as pd
    structured_df = pd.DataFrame(structured_records)
    display(structured_df.head(80))
    structured_df.to_csv(DATA / "13_structured_translation_records.csv", index=False)
except Exception:
    for rec in structured_records[:80]:
        print(rec)

## 7. Translation graph model

This notebook compares two models.

### Model A: surface mapping

```text
Language A ↔ Language B
```

### Model B: invariant correspondence

```text
Specification
      ↓
Meaning / concept
   ↙          ↘
Language A   Language B
```

Model B is the stronger form of the lab-report claim.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.axis("off")

nodes = {
    "Specification\\n(constraints)": (0.5, 0.88),
    "Meaning / Concept\\n(invariant candidate)": (0.5, 0.66),
    "Language A\\n(form A)": (0.25, 0.40),
    "Language B\\n(form B)": (0.75, 0.40),
    "Words": (0.12, 0.18),
    "Speech": (0.25, 0.18),
    "Glyphs": (0.38, 0.18),
    "Words ": (0.62, 0.18),
    "Speech ": (0.75, 0.18),
    "Glyphs ": (0.88, 0.18),
}

for label, (x, y) in nodes.items():
    ax.text(
        x, y, label,
        ha="center", va="center",
        bbox=dict(boxstyle="round,pad=0.45", linewidth=1.2),
        fontsize=10
    )

edges = [
    ("Specification\\n(constraints)", "Meaning / Concept\\n(invariant candidate)"),
    ("Meaning / Concept\\n(invariant candidate)", "Language A\\n(form A)"),
    ("Meaning / Concept\\n(invariant candidate)", "Language B\\n(form B)"),
    ("Language A\\n(form A)", "Words"),
    ("Language A\\n(form A)", "Speech"),
    ("Language A\\n(form A)", "Glyphs"),
    ("Language B\\n(form B)", "Words "),
    ("Language B\\n(form B)", "Speech "),
    ("Language B\\n(form B)", "Glyphs "),
]

for a, b in edges:
    x1, y1 = nodes[a]
    x2, y2 = nodes[b]
    ax.annotate(
        "",
        xy=(x2, y2 + 0.04),
        xytext=(x1, y1 - 0.04),
        arrowprops=dict(arrowstyle="->", linewidth=1)
    )

# Bidirectional correspondence line between language forms
ax.annotate(
    "",
    xy=(0.67, 0.40),
    xytext=(0.33, 0.40),
    arrowprops=dict(arrowstyle="<->", linewidth=1.5, linestyle="--")
)
ax.text(0.5, 0.455, "correspondence", ha="center", va="center", fontsize=9)

path = FIGURES / "13_translation_correspondence.png"
plt.savefig(path, dpi=180, bbox_inches="tight")
print("Saved:", path)
plt.show()

## 8. Invariant candidate scoring

Score each candidate invariant after inspecting actual examples.

Use:

- `preserved`
- `partially preserved`
- `changed`
- `not represented`
- `unresolved`

In [ ]:
invariant_candidates = [
    {
        "candidate": "meaning / concept",
        "score": "unresolved",
        "why_it_might_persist": "Translation is usually organized around preserving meaning.",
        "evidence": ""
    },
    {
        "candidate": "correspondence relation",
        "score": "unresolved",
        "why_it_might_persist": "Mappings can persist even when forms differ.",
        "evidence": ""
    },
    {
        "candidate": "grammar",
        "score": "unresolved",
        "why_it_might_persist": "Some structural relations may be maintained or transformed systematically.",
        "evidence": ""
    },
    {
        "candidate": "phonology",
        "score": "unresolved",
        "why_it_might_persist": "Usually changes across languages, but may preserve systematic constraints.",
        "evidence": ""
    },
    {
        "candidate": "script / glyph system",
        "score": "unresolved",
        "why_it_might_persist": "Usually changes as representation, but may preserve text-level structure.",
        "evidence": ""
    },
    {
        "candidate": "lineage",
        "score": "unresolved",
        "why_it_might_persist": "More relevant to descendant languages than direct translation.",
        "evidence": ""
    },
]

try:
    import pandas as pd
    inv_df = pd.DataFrame(invariant_candidates)
    display(inv_df)
    inv_df.to_csv(DATA / "13_invariant_candidate_scores.csv", index=False)
except Exception:
    for row in invariant_candidates:
        print(row)

## 9. Mini experiment work area

After locating a runnable translation example, fill this section with the actual command and outputs.

The target comparison is:

```text
same meaning
different language
different script / sound
```

Then answer:

1. What changed?
2. What persisted?
3. Was the invariant explicit in the specification or inferred by the engine?

In [ ]:
# TODO: replace with the smallest upstream translation command.
#
# Example shape:
#
# result = subprocess.run(
#     ["python", "..."],
#     cwd=UPSTREAM_DIR,
#     text=True,
#     capture_output=True,
#     check=True,
# )
# print(result.stdout[:2000])
#
# Then parse outputs into:
#
# source_form = ...
# target_form = ...
# meaning = ...
# correspondence = ...

## 10. Hypothesis 2

**Hypothesis 2**

Translation preserves correspondence relations while changing observable representations.

Words may change.

Scripts may change.

Pronunciations may change.

Grammar may change partially.

The strongest invariant candidates are:

- meaning,
- correspondence,
- specification constraints.

Notebook 17 should test whether writing systems preserve correspondence when the script changes.

## 11. Questions for Notebook 17

| Question | Motivation |
|---|---|
| How are scripts generated? | Connects specification to visual representation |
| What survives script changes? | Tests representation vs specification |
| Are glyphs arbitrary or constrained? | Determines whether writing systems encode structure |
| Can one meaning survive multiple scripts? | Tests correspondence preservation |